# VisionGuard — EfficientAD on MVTec AD and VisA (Colab T4)

Trains **27 separate category models**, sequentially, with training batch size **1** (required by EfficientAD).
“Batch processing” here means a queue of category jobs; it does not mean larger image minibatches.

**Changes from the supplied notebook:** removed the failing `mylee/*` URLs, corrected the VisA root,
added full Lightning training checkpoints every 500 optimizer steps and every epoch, and retained
completed models even if evaluation fails. Rerunning resumes unfinished jobs and skips completed ones.
Errors include full tracebacks and are recorded in Drive; the queue stops on the first error by default.

**Run in order on a fresh GPU runtime.** Run the install cell, restart the session if Colab requests it,
then continue from Step 2. Google Drive authorization is required.

MVTec uses a pinned public Hugging Face **mirror**, downloading one category at a time. No HF token is
required. VisA uses Amazon's official ~1.93 GB archive through Anomalib and prepares the full dataset once
per runtime. EfficientAD also downloads its pretrained teacher and ImageNette auxiliary images.
Allow at least **20 GB free local disk** and several GB in Drive for checkpoints. Colab can disconnect;
only saves already written to Drive survive, and mid-epoch resumption can repeat some samples.

**Combining later:** keep the models as a library indexed by `dataset/category` and select the matching
model at inference. Do not concatenate or average independently trained weights. A single universal
model would require a different training/evaluation setup. The final cell builds `model_index.json`.

## Step 1 — Install
Keep Colab's preinstalled matched CUDA PyTorch/torchvision pair. Do not upgrade torch independently.
This notebook targets Anomalib 2.7.0; its supported torch floor is 2.10. If the environment check below
rejects an older runtime, use a current Colab runtime before training. Restart after installation
if packages were already imported or Colab requests it.

In [ ]:
import subprocess
import sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "anomalib==2.7.0", "huggingface_hub>=0.34,<2", "packaging", "tqdm",
])
print("Install complete. If Colab requests a restart, restart now and continue at Step 2.")

## Step 2 — Check GPU and connect Drive

In [ ]:
import importlib.metadata as metadata
import json
import os
import shutil
from pathlib import Path
import torch
import torchvision
from packaging.version import Version
from google.colab import drive

if Version(torch.__version__.split("+")[0]) < Version("2.10"):
    raise RuntimeError(f"Anomalib 2.7 requires torch >= 2.10; found {torch.__version__}. Use a current Colab runtime.")
if metadata.version("anomalib") != "2.7.0":
    raise RuntimeError("Run Step 1 and restart the session before continuing.")
if not torch.cuda.is_available():
    raise RuntimeError("Choose Runtime > Change runtime type > T4 GPU, then rerun.")
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__, "torchvision:", torchvision.__version__)
drive.mount("/content/drive")

# Separate from old .pt files: those contain weights only and cannot resume training.
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/VisionGuard_Checkpoints/efficientad_v2")
LOCAL_ROOT = Path("/content/visionguard_efficientad_v2")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive output:", DRIVE_OUTPUT_DIR)
print("Local free disk (GB):", round(shutil.disk_usage(LOCAL_ROOT).free / 1e9, 1))
# Confirm Drive is writable before starting expensive jobs.
probe = DRIVE_OUTPUT_DIR / ".write_test"
probe.write_text("ok", encoding="utf-8")
probe.unlink()

## Step 3 — Choose your queue
Defaults train all 27 categories for 70 epochs each. For a quick first run, set
`SELECT_JOBS = [("mvtec", "bottle"), ("visa", "candle")]` and use a **different** `RUN_NAME`
with `MAX_EPOCHS = 1`. Do not change a saved run's configuration when resuming.
You can reduce `MAX_JOBS_PER_SESSION` to process a few categories per Colab session.

Optional existing data: set `EXISTING_MVTEC_ROOT` to the folder containing `bottle/train/good`, etc.
Set `EXISTING_VISA_ROOT` to the folder containing **`visa_pytorch`**, not to `visa_pytorch` itself.
These folders are read in place and never cleaned by this notebook. Leave both as `None` for downloads.

In [ ]:
MVTEC_CATEGORIES = [
    "bottle", "cable", "capsule", "carpet", "grid", "hazelnut", "leather",
    "metal_nut", "pill", "screw", "tile", "toothbrush", "transistor", "wood", "zipper",
]
VISA_CATEGORIES = [
    "pcb1", "pcb2", "pcb3", "pcb4", "candle", "capsules", "cashew", "chewinggum",
    "fryum", "macaroni1", "macaroni2", "pipe_fryum",
]
RUN_NAME = "full_70epochs"
MAX_EPOCHS = 70
SAVE_EVERY_N_STEPS = 500
NUM_WORKERS = 2  # Use 0 if Colab reports DataLoader worker errors.
EVAL_BATCH_SIZE = 4
SEED = 42
SELECT_JOBS = None  # None means all 27; or e.g. [("mvtec", "bottle"), ("visa", "candle")]
MAX_JOBS_PER_SESSION = None  # e.g. 2; completed jobs do not count toward this limit.
STOP_ON_ERROR = True
CLEAN_LOCAL_AFTER_SUCCESS = True
EXISTING_MVTEC_ROOT = None
EXISTING_VISA_ROOT = None

ALL_JOBS = [("mvtec", c) for c in MVTEC_CATEGORIES] + [("visa", c) for c in VISA_CATEGORIES]
JOBS = ALL_JOBS if SELECT_JOBS is None else list(SELECT_JOBS)
if not JOBS or any(job not in ALL_JOBS for job in JOBS) or len(set(JOBS)) != len(JOBS):
    raise ValueError("SELECT_JOBS must contain unique (dataset, category) pairs from ALL_JOBS.")
if Path(RUN_NAME).name != RUN_NAME or RUN_NAME in {"", ".", ".."}:
    raise ValueError("RUN_NAME must be a single folder name.")
if MAX_EPOCHS < 1 or SAVE_EVERY_N_STEPS < 1:
    raise ValueError("MAX_EPOCHS and SAVE_EVERY_N_STEPS must be positive.")
if MAX_JOBS_PER_SESSION is not None and MAX_JOBS_PER_SESSION < 1:
    raise ValueError("MAX_JOBS_PER_SESSION must be positive or None.")
RUN_DIR = DRIVE_OUTPUT_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Queue: {len(JOBS)} categories, {MAX_EPOCHS} epochs/category, training batch size 1")

## Step 4 — Dataset preparation and checkpoint helpers
VisA's official datamodule converts its CSV split and binarizes masks. Its root must be the parent
of `visa_pytorch`. Validation uses a seeded half of the test data (`from_test`); reported test metrics
use the remaining half, avoiding the default validation/test overlap. These are not full-test benchmark scores.

Progress checkpoints include optimizer, scheduler, and loop state. Two recent step saves and two
recent epoch saves are retained per category. Final `.ckpt` files support Anomalib loading; final `.pt`
files contain CPU `state_dict` weights for your existing application. Keep the configuration and versions too.

In [ ]:
import gc
import re
import traceback
from datetime import datetime, timezone
from huggingface_hub import snapshot_download
from lightning.pytorch import seed_everything
from anomalib.callbacks import ModelCheckpoint
from anomalib.data import MVTecAD, Visa
from anomalib.engine import Engine
from anomalib.engine.plugins import AnomalibCheckpointIO
from anomalib.models import EfficientAd

MVTEC_REPO = "foersben/mvtec-ad"
MVTEC_REVISION = "c75b39616f84db43677bcc8228caaafaf5096d7f"
VERSIONS = {name: metadata.version(name) for name in ["anomalib", "torch", "torchvision", "lightning"]}
VERSIONS["python"] = sys.version.split()[0]

def atomic_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)

class AtomicCheckpointIO(AnomalibCheckpointIO):
    # Keep the previous checkpoint intact if a Drive write is interrupted.
    def save_checkpoint(self, checkpoint, path, storage_options=None):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name + ".tmp")
        super().save_checkpoint(checkpoint, str(temporary), storage_options=storage_options)
        os.replace(temporary, path)
        # No validation metric is needed for periodic recovery saves. Keep the latest two
        # in each progress folder, pruning only after the new file is fully written.
        if path.parent.name in {"steps", "epochs"}:
            history = sorted(path.parent.glob("*.ckpt"), key=lambda p: p.stat().st_mtime)
            for old in history[:-2]:
                old.unlink()

def category_ready(path):
    path = Path(path)
    return (any((path / "train" / "good").glob("*.*"))
            and any((path / "test").glob("*/*.*"))
            and any((path / "ground_truth").glob("*/*.*")))

def make_datamodule(dataset, category):
    kwargs = dict(category=category, train_batch_size=1, eval_batch_size=EVAL_BATCH_SIZE,
                  num_workers=NUM_WORKERS, val_split_mode="from_test", val_split_ratio=0.5, seed=SEED)
    if dataset == "mvtec":
        root = Path(EXISTING_MVTEC_ROOT) if EXISTING_MVTEC_ROOT else LOCAL_ROOT / "datasets" / "mvtec"
        if EXISTING_MVTEC_ROOT and not category_ready(root / category):
            raise FileNotFoundError(f"Incomplete MVTec category: {root / category}")
        download_marker = root / category / ".download_complete.json"
        if not EXISTING_MVTEC_ROOT and (not download_marker.exists() or not category_ready(root / category)):
            print(f"Downloading MVTec mirror category: {category}")
            snapshot_download(repo_id=MVTEC_REPO, repo_type="dataset", revision=MVTEC_REVISION,
                              allow_patterns=[f"{category}/train/**", f"{category}/test/**",
                                              f"{category}/ground_truth/**", f"{category}/*.txt"],
                              local_dir=str(root), max_workers=4, token=False)
            if category_ready(root / category):
                atomic_json(download_marker, dict(repo=MVTEC_REPO, revision=MVTEC_REVISION))
        if not category_ready(root / category):
            raise RuntimeError(f"MVTec download incomplete: {root / category}")
        dm = MVTecAD(root=root, **kwargs)
    else:
        root = Path(EXISTING_VISA_ROOT) if EXISTING_VISA_ROOT else LOCAL_ROOT / "datasets" / "visa"
        if EXISTING_VISA_ROOT and not category_ready(root / "visa_pytorch" / category):
            raise FileNotFoundError(f"Expected prepared VisA data at {root / 'visa_pytorch' / category}")
        dm = Visa(root=root, **kwargs)  # NOT root / "visa_pytorch"
    dm.prepare_data()
    expected = root / category if dataset == "mvtec" else root / "visa_pytorch" / category
    if not category_ready(expected):
        raise RuntimeError(f"Incomplete dataset at {expected}. Remove only its incomplete local copy and retry.")
    dm.setup("fit")
    sample = next(iter(dm.train_dataloader()))
    print(f"Data ready: train={len(dm.train_data)}, val={len(dm.val_data)}, test={len(dm.test_data)}")
    print("Training sample:", tuple(sample.image.shape))
    return dm

def bind_config(job_dir, dataset, category):
    # Keep the original manifest label "s" for compatibility with existing run configs.
    # Anomalib's constructor accepts "small"; this label still denotes the same architecture.
    config = dict(dataset=dataset, category=category, model="EfficientAd", model_size="s",
                  max_epochs=MAX_EPOCHS, image_size=[256, 256], train_batch_size=1,
                  seed=SEED, validation="from_test:0.5", precision="32-true",
                  mvtec_repo=MVTEC_REPO, mvtec_revision=MVTEC_REVISION, versions=VERSIONS)
    path = job_dir / "config.json"
    if path.exists() and json.loads(path.read_text(encoding="utf-8")) != config:
        raise RuntimeError(f"Configuration/version changed for {job_dir}. Restore the original settings or use a new RUN_NAME.")
    atomic_json(path, config)
    return config

def newest_progress(job_dir):
    candidates = list((job_dir / "steps").glob("*.ckpt")) + list((job_dir / "epochs").glob("*.ckpt"))
    def step(path):
        match = re.search(r"step=(\d+)", path.name)
        return int(match.group(1)) if match else -1
    # Highest optimizer step wins, including when a session ends between epoch saves.
    return max(candidates, key=lambda p: (step(p), p.stat().st_mtime), default=None)

def make_engine(job_dir):
    callbacks = [
        ModelCheckpoint(dirpath=job_dir / "steps", filename="step={step:09d}",
                        auto_insert_metric_name=False, monitor=None, save_top_k=-1, save_last=False,
                        every_n_train_steps=SAVE_EVERY_N_STEPS, every_n_epochs=0,
                        save_on_train_epoch_end=False),
        ModelCheckpoint(dirpath=job_dir / "epochs", filename="epoch={epoch:03d}-step={step:09d}",
                        auto_insert_metric_name=False, monitor=None, save_top_k=-1, save_last=False,
                        every_n_train_steps=0, every_n_epochs=1, save_on_train_epoch_end=True),
    ]
    return Engine(default_root_dir=str(LOCAL_ROOT / "runs" / job_dir.parent.name / job_dir.name),
                  accelerator="gpu", devices=1, max_epochs=MAX_EPOCHS, precision="32-true",
                  callbacks=callbacks, plugins=[AtomicCheckpointIO()], logger=False,
                  enable_progress_bar=True, check_val_every_n_epoch=1)

def save_cpu_weights(model, path):
    temporary = path.with_name(path.name + ".tmp")
    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, temporary)
    os.replace(temporary, path)

def serializable(value):
    import math
    if isinstance(value, torch.Tensor):
        return serializable(value.detach().cpu().tolist())
    if isinstance(value, dict):
        return {str(k): serializable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [serializable(v) for v in value]
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value

def cleanup_local(dataset, category):
    # Only remove notebook-owned files beneath /content; never delete supplied data/Drive.
    targets = [LOCAL_ROOT / "runs" / dataset / category]
    if dataset == "mvtec" and not EXISTING_MVTEC_ROOT:
        targets.append(LOCAL_ROOT / "datasets" / "mvtec" / category)
    if dataset == "visa" and not EXISTING_VISA_ROOT:
        targets += [LOCAL_ROOT / "datasets" / "visa" / category,
                    LOCAL_ROOT / "datasets" / "visa" / "visa_pytorch" / category]
    for path in targets:
        resolved = path.resolve()
        if not resolved.is_relative_to(LOCAL_ROOT.resolve()) or resolved == LOCAL_ROOT.resolve():
            raise RuntimeError(f"Refusing unexpected cleanup path: {resolved}")
        if path.is_symlink():
            path.unlink()
        elif path.exists():
            shutil.rmtree(path)

## Step 5 — Train / resume
Run this cell again after reconnecting and running Steps 2–4. Do not change `RUN_NAME`, epoch budget,
seed, or package versions for an existing run. GPU training uses float32 for reliability on T4.
The same fixed epoch budget must be retained for the learning-rate schedule on resume.

`final.ckpt` is written **before testing**. If testing fails, the next run loads it and retries
evaluation without retraining. A `complete.json` marker is written only after weights and metrics
are saved. Pressing Stop preserves the latest scheduled checkpoint; abrupt disconnections cannot
save newer work. Download failures stop the queue instead of repeating the error 27 times.

In [ ]:
def run_one(dataset, category):
    seed_everything(SEED, workers=True)
    job_dir = RUN_DIR / dataset / category
    job_dir.mkdir(parents=True, exist_ok=True)
    model = engine = dm = None
    try:
        config = bind_config(job_dir, dataset, category)
        marker = job_dir / "complete.json"
        final_ckpt = job_dir / "final.ckpt"
        final_weights = job_dir / f"efficientad_{dataset}_{category}.pt"
        report = job_dir / "metrics.json"
        if marker.exists() and all(p.is_file() and p.stat().st_size > 0 for p in [final_ckpt, final_weights, report]):
            print("Already complete:", dataset, category)
            return "skipped"

        dm = make_datamodule(dataset, category)
        if final_ckpt.exists():
            print("Training finished previously; retrying evaluation:", final_ckpt)
            model = EfficientAd.load_from_checkpoint(final_ckpt)
            engine = Engine(default_root_dir=str(LOCAL_ROOT / "evaluation" / dataset / category),
                            accelerator="gpu", devices=1, logger=False)
            results = engine.test(model=model, datamodule=dm)
        else:
            model = EfficientAd(model_size="small", imagenet_dir=LOCAL_ROOT / "imagenette", visualizer=False)
            engine = make_engine(job_dir)
            progress = newest_progress(job_dir)
            print("Resuming:" if progress else "Starting fresh:", progress or f"{dataset}/{category}")
            engine.fit(model=model, datamodule=dm, ckpt_path=str(progress) if progress else None)
            # Save full training state even if test/evaluation fails later.
            engine.trainer.save_checkpoint(str(final_ckpt))
            # fit already calibrated on validation; reuse the trainer without rebuilding workspace paths.
            results = engine.trainer.test(model=model, datamodule=dm, ckpt_path=None)

        # Save calibrated weights after validation/test setup, preserving the app's state_dict format.
        save_cpu_weights(model, final_weights)
        atomic_json(report, dict(dataset=dataset, category=category, metrics=serializable(results),
                                 test_protocol="seeded 50% holdout of original test set", config=config))
        atomic_json(marker, dict(dataset=dataset, category=category, checkpoint="final.ckpt",
                                 weights=final_weights.name, metrics="metrics.json",
                                 finished_utc=datetime.now(timezone.utc).isoformat()))
        (job_dir / "error.json").unlink(missing_ok=True)
        if CLEAN_LOCAL_AFTER_SUCCESS:
            cleanup_local(dataset, category)
        print("Saved:", final_ckpt, "and", final_weights)
        return "completed"
    except Exception:
        error = traceback.format_exc()
        print(error)
        atomic_json(job_dir / "error.json", dict(dataset=dataset, category=category, traceback=error,
                                                  time_utc=datetime.now(timezone.utc).isoformat()))
        raise
    finally:
        # Trainer and model reference each other, so release both before collecting CUDA tensors.
        del model, engine, dm
        gc.collect()
        torch.cuda.empty_cache()

summary = {}
attempted = 0
for dataset, category in JOBS:
    if MAX_JOBS_PER_SESSION is not None and attempted >= MAX_JOBS_PER_SESSION:
        print("Session job limit reached. Rerun later to continue.")
        break
    print(f"\n=== {dataset.upper()} / {category} ===")
    try:
        status = run_one(dataset, category)
        summary[f"{dataset}/{category}"] = status
        attempted += status != "skipped"
    except Exception:
        summary[f"{dataset}/{category}"] = "failed — see error.json"
        attempted += 1
        if STOP_ON_ERROR:
            print("Queue stopped. Fix the error above, then rerun; saved progress remains in Drive.")
            break
    except KeyboardInterrupt:
        print("Stopped by user. Rerun to resume the latest scheduled checkpoint.")
        break
print("\nSession results:", json.dumps(summary, indent=2))
print("Checkpoints:", RUN_DIR)

## Step 6 — Inspect results and build the model library index
This scans all completed models in this run, including previous sessions. Copy the whole run folder
to deploy the library. Use each model only for its trained category. Both the full `.ckpt` and the
weights-only `.pt` contain more than student weights: category statistics and calibration matter.
Loading `.pt` requires the same architecture, preprocessing, and compatible library version.

In [ ]:
index = {}
for dataset, category in ALL_JOBS:
    folder = RUN_DIR / dataset / category
    marker = folder / "complete.json"
    if not marker.exists():
        continue
    info = json.loads(marker.read_text(encoding="utf-8"))
    if not all((folder / info[key]).is_file() for key in ["checkpoint", "weights", "metrics"]):
        print("Incomplete files:", dataset, category)
        continue
    index[f"{dataset}/{category}"] = {
        key: str((folder / info[key]).relative_to(RUN_DIR)) for key in ["checkpoint", "weights", "metrics"]
    }
atomic_json(RUN_DIR / "model_index.json", index)
print(f"Completed: {len(index)}/27")
for key, paths in index.items():
    print(key, "->", paths["checkpoint"])
print("Library index:", RUN_DIR / "model_index.json")

## Loading one completed model later (optional)
This uses the full Anomalib checkpoint. Train/validation data are not needed to load it.
The first validation during training establishes category calibration; keep that state intact.

In [ ]:
# Example (uncomment after the bottle category finishes):
# checkpoint = RUN_DIR / "mvtec" / "bottle" / "final.ckpt"
# detector = EfficientAd.load_from_checkpoint(checkpoint, map_location="cpu")
# detector.eval()
# For deployment inference use Engine.predict with this loaded model and your input dataset.
# Route each image to the detector for its known dataset/category using model_index.json.

## Sources and troubleshooting
- [EfficientAD implementation, Anomalib 2.7.0](https://github.com/open-edge-platform/anomalib/blob/lib/v2.7.0/src/anomalib/models/image/efficient_ad/lightning_model.py)
- [VisA datamodule and official archive URL](https://github.com/open-edge-platform/anomalib/blob/lib/v2.7.0/src/anomalib/data/datamodules/image/visa.py)
- [MVTec AD official dataset](https://www.mvtec.com/research-teaching/datasets/mvtec-ad)
- [MVTec mirror used here](https://huggingface.co/datasets/foersben/mvtec-ad/tree/c75b39616f84db43677bcc8228caaafaf5096d7f)

If the mirror becomes unavailable, provide your extracted MVTec folder through `EXISTING_MVTEC_ROOT`.
No token or disabled TLS checks are required. Keep source dataset license files with your datasets.
If Drive fills up, remove **older completed runs you no longer need**, or move them elsewhere;
the notebook retains progress saves for recovery. A final checkpoint can be much larger than 15 MB.
If a scheduled checkpoint is damaged, the traceback identifies it; move that file aside and rerun
to use the next retained checkpoint. `.tmp` files are ignored when resuming.

The notebook was statically validated and its checkpoint/data helpers tested locally. Full training
and Google Drive writes must be verified in your authenticated Colab T4 session.